### 사용 전, 반드시 rds_db_car.py의 import 부분 변경
<모듈로 쓸때>
```
from .rds_db_setup import get_or_create_async_db_engine
```
<주피터 쓸때>
```
from rds_db_setup import get_or_create_async_db_engine
```

In [2]:
import os
import sys
from pathlib import Path
import pandas as pd
import asyncio
from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_postgres.vectorstores import PGVector

from sqlalchemy import select, inspect, text, delete


project_root = Path.cwd().parent

# 파이썬 모듈 검색 경로에 프로젝트 루트를 추가합니다.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.rds_schema_builder import qna_history_schema
from utils.rds_db_setup import get_or_create_async_db_engine
from utils.rds_db_car import all_pipeline, get_car_documents_table, unique_filter_load


# 현재 노트북 파일의 위치(utils/)에서 두 단계 상위 폴더(rag/)로 이동하여
# 프로젝트 루트 디렉토리를 찾습니다.


load_dotenv()
os.environ["LANGCHAIN_PROJECT"] = "HY RAG Project"

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(temperature=0, model="gpt-5-mini") 

db_engine = await get_or_create_async_db_engine("hy_rag_db",False)
DB_NAME = "hy_rag_db"
car_table = "car_documents"

# 메타데이터, 스키마 호출
meta_obj, admission_stats_table = get_car_documents_table()
qna_meta, qna_schema = qna_history_schema()
# csv 데이터 저장
# await all_pipeline()


car_vectorstore = PGVector(
    embeddings=embeddings,
    collection_name=car_table,
    connection=db_engine,   
)


In [3]:
"""
데이터베이스에 있는 모든 테이블의 이름을 조회하여 출력합니다.
"""
async def list_all_tables():
    db_engine = await get_or_create_async_db_engine(DB_NAME, install_pgvector=True)

    try:
        async with db_engine.connect() as conn:
            table_names = await conn.run_sync(
                lambda sync_conn: inspect(sync_conn).get_table_names()
            )

        if table_names:
            print(f"총 {len(table_names)}개의 테이블을 찾았습니다.")
            for name in table_names:
                print(f"  - {name}")
        else:
            print("ℹ️ 데이터베이스에 테이블이 존재하지 않습니다.")

    except Exception as e:
        print(f"❌ 테이블 조회 중 오류 발생: {e}")
    finally:
        await db_engine.dispose()


await list_all_tables()

'hy_rag_db' 데이터베이스에 'vector' 확장이 활성화되었습니다.
총 3개의 테이블을 찾았습니다.
  - car_documents
  - qna_summary
  - qna_chat_history


In [4]:
'''
데이터 조회 및 개수 세기

schema_name: 조회할 테이블의 스키마를 넣어주면 조회 가능

'''

async def view_car_documents_table(schema_name: str = admission_stats_table):
    db_engine = await get_or_create_async_db_engine(DB_NAME, install_pgvector=False)

    try:
        async with db_engine.connect() as conn:
            select_stmt = select(schema_name)
            result = await conn.execute(select_stmt)
            records = result.mappings().all()
            records_num = records

        if records:
            print(f"테이블에서 총 {len(records_num)}개 데이터를 조회했습니다.")
            df = pd.DataFrame(records)
            column_order = [c.name for c in schema_name.columns]
            df = df[column_order]
            display(df.tail(10))
            return df
        else:
            print("ℹ 테이블에 데이터가 없습니다.")
            return None
    except Exception as e:
        print(f"❌ 테이블 조회 중 오류 발생: {e}")
        return None
    finally:
        await db_engine.dispose()

df_result = await view_car_documents_table(schema_name = qna_schema) 

테이블에서 총 14711개 데이터를 조회했습니다.


,id,session_id,role,content,created_at
14701,18170,test,ai,이화여자대학교의 12년 특례 전형에 대해 안내해드릴게요! 😊\n\n### 12년 특...,2025-11-12 02:38:29.325907
14702,18175,test,human,이대 12년 특례 알려주세요,2025-11-12 02:38:31.584889
14703,18179,test,human,이대 12년 특례 알려주세요,2025-11-12 02:38:31.829654
14704,18182,test,ai,이화여자대학교의 12년 특례 전형에 대해 안내해드릴게요! 😊\n\n### 12년 특...,2025-11-12 02:38:31.959693
14705,18168,test,ai,이화여자대학교의 12년 특례 전형에 대해 안내해드릴게요! 😊\n\n### 12년 특...,2025-11-12 02:38:29.143295
14706,18173,test,human,이대 12년 특례 알려주세요,2025-11-12 02:38:31.332293
14707,18178,test,ai,이화여자대학교의 12년 특례 전형에 대해 안내해드릴게요! 😊\n\n### 12년 특...,2025-11-12 02:38:31.746705
14708,18181,test,human,이대 12년 특례 알려주세요,2025-11-12 02:38:31.934115
14709,18176,test,ai,이화여자대학교의 12년 특례 전형에 대해 안내해드릴게요! 😊\n\n### 12년 특...,2025-11-12 02:38:31.601280
14710,18180,test,ai,이화여자대학교의 12년 특례 전형에 대해 안내해드릴게요! 😊\n\n### 12년 특...,2025-11-12 02:38:31.849121


In [ ]:
# 형식이 맞지 않는 세션 아이디 리스트 확인
session_ids = df_result['session_id'].value_counts()
session_id_series = session_ids[session_ids.index.str.len() != 36]
session_id_list = session_id_series.index
print(session_id_list)

In [ ]:
'''
조건별 데이터 삭제하기 (table, column, 삭제할 category)
'''
async def delete_specific_data(table, columns, cat_name: str):
    try:
        async with db_engine.begin() as conn:
            sql_query = text(f"DELETE FROM {table} WHERE {columns} = :cat_name")
            result = await conn.execute(sql_query, {"cat_name": cat_name})
            print(f"{result.rowcount}개의 '{cat_name}' 데이터가 삭제되었습니다.")
            
    except Exception as e:
        print(f"❌ 데이터 삭제 중 오류 발생: {e}")
    finally:
        await db_engine.dispose()

In [ ]:
# 세션 아이디 제거
for session_id in session_id_list:
    await delete_specific_data(table = 'qna_chat_history', columns='session_id', cat_name=session_id)

In [ ]:
# chat_history db의 session_id 삭제
async def delete_session_by_id(session_id: str):
    try:
        async with db_engine.begin() as conn:
            sql_query = text("DELETE FROM qna_chat_history WHERE session_id = :sid") # qna_chat_history가 테이블이름
            result = await conn.execute(sql_query, {"sid": session_id})
            print(f"{result.rowcount}개의 '{session_id}' 세션 데이터가 삭제되었습니다.")
    except Exception as e:
        print(f"❌ 데이터 삭제 중 오류 발생: {e}")

await delete_session_by_id("테스트")

2개의 '테스트' 세션 데이터가 삭제되었습니다.


In [4]:
"""
car_documents 테이블의 모든 데이터 삭제
"""
async def delete_all_data(table_name :str):
    try:
        async with db_engine.begin() as conn:
            sql_query = text(f"TRUNCATE TABLE {table_name} RESTART IDENTITY;")
            await conn.execute(sql_query)
            print("'car_documents' 테이블의 모든 데이터가 삭제되었습니다.")
            
    except Exception as e:
        print(f"❌ 데이터 삭제 중 오류 발생: {e}")
    finally:
        await db_engine.dispose()
        
await delete_all_data('qna_chat_history')

'car_documents' 테이블의 모든 데이터가 삭제되었습니다.


In [23]:
# SQLDB테이블 자체 삭제
async def drop_table(engine, table_name: str):
    """
    지정된 테이블 자체를 삭제
    """
    try:
        async with engine.begin() as conn:
            sql_query = text(f'DROP TABLE IF EXISTS "{table_name}" CASCADE;')
            await conn.execute(sql_query)
        print(f"✅ '{table_name}' 테이블이 삭제되었습니다.")
    except Exception as e:
        print(f"❌ 테이블 삭제 중 오류 발생: {e}")

await drop_table(db_engine, 'none')

✅ 'none' 테이블이 삭제되었습니다.


In [ ]:
# VectorDB 컬렉션 및 관련 임베딩 삭제
async def delete_collection(engine, coll_name: str, force_no_cascade: bool = False):
    """
    LangChain PGVector 컬렉션(coll_name)을 삭제.
    FK에 ON DELETE CASCADE가 걸려 있으면 컬렉션만 지워도 임베딩이 함께 삭제됨.
    force_no_cascade=True면 수동으로 임베딩부터 삭제.
    """
    async with engine.begin() as conn:
        # 1) 컬렉션 ID 조회
        coll_id = await conn.scalar(
            text("SELECT uuid FROM langchain_pg_collection WHERE name = :name"),
            {"name": coll_name},
        )
        if not coll_id:
            print(f"⚠️ 컬렉션 '{coll_name}'을 찾을 수 없습니다.")
            return

        if force_no_cascade:
            # 2-a) (옵션) CASCADE가 없다면 임베딩부터 수동 삭제
            await conn.execute(
                text("DELETE FROM langchain_pg_embedding WHERE collection_id = :cid"),
                {"cid": coll_id},
            )

        # 2-b) 컬렉션 삭제 (CASCADE면 여기 한 줄로 끝)
        await conn.execute(
            text("DELETE FROM langchain_pg_collection WHERE uuid = :cid"),
            {"cid": coll_id},
        )

    print(f"✅ 컬렉션 '{coll_name}' 및 관련 임베딩이 삭제되었습니다.")

await delete_collection(db_engine, "qna_documents")

⚠️ 컬렉션 'qna_chat_history'을 찾을 수 없습니다.
